In [1]:
import os
import json
import traceback
from datetime import datetime
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType, TimestampType
)
from pyspark.sql.window import Window

# ── Konfigurasi HDFS ──────────────────────────────────────────────────────────
HDFS_HOST      = "100.74.49.87"
HDFS_PORT      = 8020
HDFS_BASE      = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"
HDFS_API_DIR   = f"{HDFS_BASE}/data/saham/api/"
HDFS_RSS_DIR   = f"{HDFS_BASE}/data/saham/rss/"
HDFS_HASIL_DIR = f"{HDFS_BASE}/data/saham/hasil/"

# ── Output lokal ──────────────────────────────────────────────────────────────
DASHBOARD_DIR  = Path("../dashboard/data")
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON    = DASHBOARD_DIR / "spark_results.json"

# ── Perusahaan target untuk Analisis 3 ────────────────────────────────────────
TARGET_COMPANIES = {
    "BCA"    : ["BCA", "Bank Central Asia"],
    "BRI"    : ["BRI", "Bank Rakyat Indonesia"],
    "Telkom" : ["Telkom", "TLKM"],
    "Astra"  : ["Astra", "ASII"],
    "Mandiri": ["Mandiri", "Bank Mandiri"],
}

print("✅ Import selesai")
print(f"📁 Dashboard output : {OUTPUT_JSON.resolve()}")
print(f"🌐 HDFS Base        : {HDFS_BASE}")

✅ Import selesai
📁 Dashboard output : /Users/gilang/Documents/Kuliah/Semester 4/Big Data/kelompok-3-ets-bigdata/dashboard/data/spark_results.json
🌐 HDFS Base        : hdfs://100.74.49.87:8020


In [2]:
spark = (
    SparkSession.builder
    .appName("SahamMeter-ETS-Kelompok3")
    # ── HDFS DataNode routing ──────────────────────────────────────────────
    .config("spark.hadoop.dfs.datanode.hostname",                "100.74.49.87")
    .config("spark.hadoop.dfs.client.use.datanode.hostname",     "true")
    .config("spark.hadoop.dfs.datanode.use.datanode.hostname",   "true")
    # ── Override IP internal Docker → IP Tailscale ─────────────────────────
    .config("spark.hadoop.fs.hdfs.impl",
            "org.apache.hadoop.hdfs.DistributedFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.hdfs.impl",
            "org.apache.hadoop.fs.Hdfs")
    # ── Timeout lebih pendek (ms) ──────────────────────────────────────────
    .config("spark.hadoop.dfs.client.socket-timeout",            "10000")
    .config("spark.hadoop.ipc.client.connect.timeout",           "10000")
    .config("spark.network.timeout",                             "120s")
    # ── JSON & parsing ─────────────────────────────────────────────────────
    .config("spark.sql.legacy.timeParserPolicy",                 "LEGACY")
    .config("spark.sql.jsonGenerator.ignoreNullFields",          "false")
    # ── Misc ───────────────────────────────────────────────────────────────
    .config("spark.hadoop.fs.hdfs.impl.disable.cache",           "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession aktif : {spark.version}")
print(f"   App Name          : {spark.sparkContext.appName}")
print(f"   Master            : {spark.sparkContext.master}")

26/05/03 14:36:23 WARN Utils: Your hostname, Gilangs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.4.65.87 instead (on interface en0)
26/05/03 14:36:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 14:36:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 14:36:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ SparkSession aktif : 3.5.1
   App Name          : SahamMeter-ETS-Kelompok3
   Master            : local[*]


In [3]:
def hdfs_list_files(hdfs_path: str) -> list[str]:
    """List semua file di direktori HDFS. Return list path string."""
    try:
        jvm    = spark._jvm
        conf   = spark._jsc.hadoopConfiguration()
        fs     = jvm.org.apache.hadoop.fs.FileSystem.get(
                     jvm.java.net.URI(hdfs_path), conf)
        status = fs.listStatus(jvm.org.apache.hadoop.fs.Path(hdfs_path))
        paths  = [s.getPath().toString() for s in status]
        return paths
    except Exception as e:
        print(f"⚠️  HDFS list gagal untuk {hdfs_path}: {e}")
        return []


def hdfs_path_exists(hdfs_path: str) -> bool:
    """Cek apakah path HDFS ada."""
    try:
        jvm  = spark._jvm
        conf = spark._jsc.hadoopConfiguration()
        fs   = jvm.org.apache.hadoop.fs.FileSystem.get(
                   jvm.java.net.URI(hdfs_path), conf)
        return fs.exists(jvm.org.apache.hadoop.fs.Path(hdfs_path))
    except Exception as e:
        print(f"⚠️  HDFS exists check gagal: {e}")
        return False


# ── Test koneksi ───────────────────────────────────────────────────────────────
print("🔍 Test koneksi HDFS...")
for label, path in [("API", HDFS_API_DIR), ("RSS", HDFS_RSS_DIR), ("Hasil", HDFS_HASIL_DIR)]:
    exists = hdfs_path_exists(path)
    files  = hdfs_list_files(path) if exists else []
    status = f"✅ Ada, {len(files)} file" if exists else "❌ Tidak ditemukan"
    print(f"   [{label:5s}] {path}  →  {status}")
    for f in files[:5]:
        print(f"            └─ {f.split('/')[-1]}")
    if len(files) > 5:
        print(f"            └─ ... dan {len(files)-5} file lainnya")

🔍 Test koneksi HDFS...
   [API  ] hdfs://100.74.49.87:8020/data/saham/api/  →  ✅ Ada, 5 file
            └─ 2026-04-30_01-21-57.json
            └─ 2026-04-30_01-27-05.json
            └─ 2026-04-30_01-34-42.json
            └─ 2026-04-30_01-39-44.json
            └─ 2026-04-30_01-48-37.json
   [RSS  ] hdfs://100.74.49.87:8020/data/saham/rss/  →  ✅ Ada, 3 file
            └─ 2026-04-30_01-22-04.json
            └─ 2026-04-30_01-34-43.json
            └─ 2026-04-30_01-48-38.json
   [Hasil] hdfs://100.74.49.87:8020/data/saham/hasil/  →  ✅ Ada, 1 file
            └─ _spark_metadata


In [12]:
schema_saham = StructType([
    StructField("ticker",       StringType(),  True),
    StructField("harga",        DoubleType(),  True),
    StructField("open",         DoubleType(),  True),
    StructField("high",         DoubleType(),  True),
    StructField("low",          DoubleType(),  True),
    StructField("volume",       LongType(),    True),
    StructField("change_pct",   DoubleType(),  True),
    StructField("is_simulated", StringType(),  True),
    StructField("timestamp",    StringType(),  True),
])

schema_berita = StructType([
    StructField("id",           StringType(), True),
    StructField("judul",        StringType(), True),
    StructField("ringkasan",    StringType(), True),
    StructField("sentimen",     StringType(), True),
    StructField("sumber",       StringType(), True),
    StructField("timestamp",    StringType(), True),
    StructField("url",          StringType(), True),
    StructField("waktu_terbit", StringType(), True),
])

print("✅ Schema didefinisikan")
spark.createDataFrame([], schema_saham).printSchema()
spark.createDataFrame([], schema_berita).printSchema()

✅ Schema didefinisikan
root
 |-- ticker: string (nullable = true)
 |-- harga: double (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- change_pct: double (nullable = true)
 |-- is_simulated: string (nullable = true)
 |-- timestamp: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- judul: string (nullable = true)
 |-- ringkasan: string (nullable = true)
 |-- sentimen: string (nullable = true)
 |-- sumber: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- url: string (nullable = true)
 |-- waktu_terbit: string (nullable = true)



In [8]:
print("📂 Membaca data saham dari HDFS...")
df_saham = None
SAHAM_DARI_HDFS = False

try:
    if not hdfs_path_exists(HDFS_API_DIR):
        raise FileNotFoundError(f"Path tidak ditemukan: {HDFS_API_DIR}")

    file_list  = hdfs_list_files(HDFS_API_DIR)
    json_files = [f for f in file_list if f.endswith(".json")]

    if not json_files:
        raise ValueError("Tidak ada file .json di direktori HDFS API")

    print(f"   Ditemukan {len(json_files)} file JSON")

    df_saham = (
        spark.read
        .option("multiLine", "true")
        .option("mode", "PERMISSIVE")
        .schema(schema_saham)
        .json(HDFS_API_DIR + "*.json")
    )

    count = df_saham.count()
    if count == 0:
        raise ValueError("File JSON ada tapi tidak ada baris data")

    SAHAM_DARI_HDFS = True
    print(f"   ✅ Berhasil baca {count} baris data saham dari HDFS")

except Exception as e:
    print(f"   ❌ Gagal baca dari HDFS: {e}")
    df_saham = buat_dummy_saham(spark)

# Sesuaikan dropna dengan nama field asli
df_saham = df_saham.dropna(subset=["ticker", "harga", "open"])

print(f"\n📊 Preview data saham ({'HDFS' if SAHAM_DARI_HDFS else 'DUMMY'}):")
df_saham.show(10, truncate=False)

📂 Membaca data saham dari HDFS...
   Ditemukan 5 file JSON
   ✅ Berhasil baca 50 baris data saham dari HDFS

📊 Preview data saham (HDFS):
+------+--------+--------+-------+-------+-------+----------+------------+--------------------------+
|ticker|harga   |open    |high   |low    |volume |change_pct|is_simulated|timestamp                 |
+------+--------+--------+-------+-------+-------+----------+------------+--------------------------+
|BBCA  |9577.66 |9588.45 |9604.1 |9510.67|3226870|-0.1125   |true        |2026-04-30T01:44:18.086894|
|BBRI  |4783.64 |4809.24 |4814.99|4756.41|1505290|-0.5323   |true        |2026-04-30T01:44:18.490470|
|TLKM  |3925.12 |3873.91 |3938.93|3871.16|4301228|1.3219    |true        |2026-04-30T01:44:18.553681|
|ASII  |5249.9  |5148.15 |5275.36|5145.94|4754494|1.9764    |true        |2026-04-30T01:44:18.615540|
|BMRI  |6051.88 |6143.93 |6167.31|6042.57|2930974|-1.4982   |true        |2026-04-30T01:44:18.671154|
|UNVR  |2079.9  |2104.94 |2109.41|2079.55|2659

In [6]:
# Baca tanpa schema dulu, biar Spark auto-detect
df_raw = spark.read.option("multiLine", "true").json(HDFS_API_DIR + "*.json")
df_raw.printSchema()
df_raw.show(3, truncate=False)


root
 |-- change_pct: double (nullable = true)
 |-- harga: double (nullable = true)
 |-- high: double (nullable = true)
 |-- is_simulated: boolean (nullable = true)
 |-- low: double (nullable = true)
 |-- open: double (nullable = true)
 |-- ticker: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- volume: long (nullable = true)

+----------+-------+-------+------------+-------+-------+------+--------------------------+-------+
|change_pct|harga  |high   |is_simulated|low    |open   |ticker|timestamp                 |volume |
+----------+-------+-------+------------+-------+-------+------+--------------------------+-------+
|-0.1125   |9577.66|9604.1 |true        |9510.67|9588.45|BBCA  |2026-04-30T01:44:18.086894|3226870|
|-0.5323   |4783.64|4814.99|true        |4756.41|4809.24|BBRI  |2026-04-30T01:44:18.490470|1505290|
|1.3219    |3925.12|3938.93|true        |3871.16|3873.91|TLKM  |2026-04-30T01:44:18.553681|4301228|
+----------+-------+-------+------------+-------

In [9]:
df_agg = df_saham.groupBy("ticker").agg(
    F.first("open",  ignorenulls=True).alias("harga_awal"),
    F.last("harga",  ignorenulls=True).alias("harga_terkini"),
    F.max("high").alias("harga_tertinggi"),
    F.min("low").alias("harga_terendah"),
    F.sum("volume").alias("total_volume"),
)

df_return = df_agg.withColumn(
    "return_pct",
    F.round(
        (F.col("harga_terkini") - F.col("harga_awal")) / F.col("harga_awal") * 100,
        4
    )
).withColumn(
    "status",
    F.when(F.col("return_pct") > 0, "NAIK")
     .when(F.col("return_pct") < 0, "TURUN")
     .otherwise("FLAT")
).orderBy(F.col("return_pct").desc())

df_return.select(
    "ticker", "harga_awal", "harga_terkini", "return_pct", "status", "total_volume"
).show(truncate=False)

result_return = df_return.toPandas().to_dict(orient="records")
print(f"✅ Analisis 1 selesai — {len(result_return)} saham diproses")

+------+----------+-------------+----------+------+------------+
|ticker|harga_awal|harga_terkini|return_pct|status|total_volume|
+------+----------+-------------+----------+------+------------+
|INDF  |7130.67   |7304.04      |2.4313    |NAIK  |24435923    |
|ASII  |5148.15   |5251.12      |2.0001    |NAIK  |24583782    |
|ICBP  |11545.97  |11655.09     |0.9451    |NAIK  |19026414    |
|TLKM  |3873.91   |3895.08      |0.5465    |NAIK  |32603401    |
|GOTO  |87.74     |87.98        |0.2735    |NAIK  |24172205    |
|BMRI  |6143.93   |6133.44      |-0.1707   |TURUN |28199297    |
|UNVR  |2104.94   |2096.02      |-0.4238   |TURUN |24791960    |
|PGAS  |1461.23   |1448.01      |-0.9047   |TURUN |34390053    |
|BBRI  |4809.24   |4758.6       |-1.053    |TURUN |20855132    |
|BBCA  |9588.45   |9415.75      |-1.8011   |TURUN |25166253    |
+------+----------+-------------+----------+------+------------+

✅ Analisis 1 selesai — 10 saham diproses


In [10]:
df_volatilitas = df_saham.groupBy("ticker").agg(
    F.round(F.stddev("harga"), 4).alias("stddev_harga"),
    F.round(F.avg("harga"), 4).alias("rata_rata_harga"),
    F.max("high").alias("high"),
    F.min("low").alias("low"),
    F.count("*").alias("jumlah_snapshot"),
).withColumn(
    "range_intraday",
    F.round(F.col("high") - F.col("low"), 2)
).withColumn(
    "cv_pct",
    F.round(
        F.when(F.col("rata_rata_harga") > 0,
               F.col("stddev_harga") / F.col("rata_rata_harga") * 100)
         .otherwise(0.0),
        4
    )
).withColumn(
    "level_volatilitas",
    F.when(F.col("cv_pct") > 2.0, "TINGGI")
     .when(F.col("cv_pct") > 0.5, "SEDANG")
     .otherwise("RENDAH")
).orderBy(F.col("cv_pct").desc())

df_volatilitas.show(truncate=False)

result_volatilitas = df_volatilitas.toPandas().fillna(0).to_dict(orient="records")
print(f"✅ Analisis 2 selesai — {len(result_volatilitas)} saham diproses")

+------+------------+---------------+--------+-------+---------------+--------------+------+-----------------+
|ticker|stddev_harga|rata_rata_harga|high    |low    |jumlah_snapshot|range_intraday|cv_pct|level_volatilitas|
+------+------------+---------------+--------+-------+---------------+--------------+------+-----------------+
|PGAS  |27.0451     |1461.414       |1503.68 |1433.88|5              |69.8          |1.8506|SEDANG           |
|ICBP  |165.5878    |11586.426      |11819.64|11293.8|5              |525.84        |1.4292|SEDANG           |
|INDF  |94.3632     |7180.122       |7336.56 |7060.46|5              |276.1         |1.3142|SEDANG           |
|UNVR  |26.3422     |2108.47        |2162.11 |2064.34|5              |97.77         |1.2494|SEDANG           |
|TLKM  |45.09       |3891.708       |3962.62 |3812.55|5              |150.07        |1.1586|SEDANG           |
|BBCA  |99.9134     |9409.992       |9604.1  |9252.17|5              |351.93        |1.0618|SEDANG           |
|

In [13]:
def buat_dummy_berita(spark):
    print("   ⚠️  Menggunakan data DUMMY untuk berita")
    dummy = [
        ("1", "BCA catat laba bersih Rp 48 triliun", "", "positif", "CNBC", "2025-04-01", "https://cnbc.id/1", "2025-04-01"),
        ("2", "BRI ekspansi kredit UMKM", "", "positif", "Bisnis", "2025-04-01", "https://bisnis.id/1", "2025-04-01"),
        ("3", "Telkom luncurkan layanan 5G", "", "positif", "Detik", "2025-04-01", "https://detik.id/1", "2025-04-01"),
        ("4", "Astra genjot penjualan EV", "", "netral", "Kontan", "2025-04-01", "https://kontan.id/1", "2025-04-01"),
        ("5", "Mandiri syariah berencana IPO", "", "positif", "Tempo", "2025-04-01", "https://tempo.id/1", "2025-04-01"),
    ]
    return spark.createDataFrame(dummy, schema=schema_berita)

print("📰 Membaca data berita dari HDFS...")
df_berita = None
BERITA_DARI_HDFS = False

try:
    if not hdfs_path_exists(HDFS_RSS_DIR):
        raise FileNotFoundError(f"Path tidak ditemukan: {HDFS_RSS_DIR}")

    file_list  = hdfs_list_files(HDFS_RSS_DIR)
    json_files = [f for f in file_list if f.endswith(".json")]

    if not json_files:
        raise ValueError("Tidak ada file .json di direktori HDFS RSS")

    print(f"   Ditemukan {len(json_files)} file JSON")

    df_berita = (
        spark.read
        .option("multiLine", "true")
        .option("mode", "PERMISSIVE")
        .schema(schema_berita)
        .json(HDFS_RSS_DIR + "*.json")
    )

    count = df_berita.count()
    if count == 0:
        raise ValueError("File JSON ada tapi tidak ada baris data")

    BERITA_DARI_HDFS = True
    print(f"   ✅ Berhasil baca {count} baris data berita dari HDFS")

except Exception as e:
    print(f"   ❌ Gagal baca dari HDFS: {e}")
    df_berita = buat_dummy_berita(spark)

df_berita = df_berita.dropna(subset=["judul"])

print(f"\n📰 Preview data berita ({'HDFS' if BERITA_DARI_HDFS else 'DUMMY'}):")
df_berita.select("judul", "sentimen", "sumber", "waktu_terbit").show(10, truncate=55)

📰 Membaca data berita dari HDFS...
   Ditemukan 3 file JSON


   ✅ Berhasil baca 400 baris data berita dari HDFS

📰 Preview data berita (HDFS):
+-------------------------------------------------------+--------+--------------------------------------+-------------------------------+
|                                                  judul|sentimen|                                sumber|                   waktu_terbit|
+-------------------------------------------------------+--------+--------------------------------------+-------------------------------+
|Tips Pilih Perlindungan Kendaraan di Era Mobil Konve...|  netral|CNN Indonesia | Berita Terkini Ekonomi|Thu, 30 Apr 2026 00:02:18 +0700|
|Bank Mega Salurkan Bantuan Sembako ke Korban Bencana...|  netral|CNN Indonesia | Berita Terkini Ekonomi|Wed, 29 Apr 2026 23:15:33 +0700|
|  Iran Resmi Larang Ekspor Baja Gegara Agresi AS-Israel|  netral|CNN Indonesia | Berita Terkini Ekonomi|Wed, 29 Apr 2026 22:00:55 +0700|
|TASPEN Beri Santunan ke Ahli Waris Guru Korban Kecel...|  netral|CNN Indonesia | Berita T

In [14]:
print("━" * 60)
print("📰 ANALISIS 3: Frekuensi Sebutan Perusahaan di Berita")
print("━" * 60)

df_tagged = df_berita

for company, keywords in TARGET_COMPANIES.items():
    pattern  = "|".join(keywords)
    col_name = f"mention_{company.lower()}"
    df_tagged = df_tagged.withColumn(
        col_name,
        F.when(
            F.regexp_extract(F.upper(F.col("judul")), pattern.upper(), 0) != "",
            1
        ).otherwise(0)
    )

print("\nSample tagging berita:")
df_tagged.select(
    "judul",
    *[f"mention_{c.lower()}" for c in TARGET_COMPANIES]
).show(10, truncate=50)

# Hitung total mention per perusahaan
mention_counts = {}
for company in TARGET_COMPANIES:
    col_name = f"mention_{company.lower()}"
    count = df_tagged.agg(F.sum(col_name).cast("long")).collect()[0][0] or 0
    mention_counts[company] = int(count)

# Buat DataFrame hasil
rows_frekuensi = [
    (company, int(count), list(TARGET_COMPANIES[company]))
    for company, count in sorted(mention_counts.items(), key=lambda x: -x[1])
]

df_frekuensi = spark.createDataFrame(
    [(r[0], r[1]) for r in rows_frekuensi],
    schema=StructType([
        StructField("perusahaan",     StringType(), True),
        StructField("jumlah_sebutan", LongType(),   True),
    ])
)

print("\n🏆 Peringkat sebutan perusahaan di berita:")
df_frekuensi.show()

print("\n📊 Distribusi sentimen keseluruhan berita:")
df_berita.groupBy("sentimen").count().orderBy(F.col("count").desc()).show()

result_frekuensi = [
    {"perusahaan": r[0], "jumlah_sebutan": r[1], "keywords": r[2]}
    for r in rows_frekuensi
]

print(f"✅ Analisis 3 selesai")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📰 ANALISIS 3: Frekuensi Sebutan Perusahaan di Berita
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Sample tagging berita:


+--------------------------------------------------+-----------+-----------+--------------+-------------+---------------+
|                                             judul|mention_bca|mention_bri|mention_telkom|mention_astra|mention_mandiri|
+--------------------------------------------------+-----------+-----------+--------------+-------------+---------------+
|Tips Pilih Perlindungan Kendaraan di Era Mobil ...|          0|          0|             0|            0|              0|
|Bank Mega Salurkan Bantuan Sembako ke Korban Be...|          0|          0|             0|            0|              0|
|Iran Resmi Larang Ekspor Baja Gegara Agresi AS-...|          0|          0|             0|            0|              0|
|TASPEN Beri Santunan ke Ahli Waris Guru Korban ...|          0|          0|             0|            0|              0|
|PAM Jaya Minta Rumah Tersambung Pipa PDAM Setop...|          0|          0|             0|            0|              0|
|                       


🏆 Peringkat sebutan perusahaan di berita:


+----------+--------------+
|perusahaan|jumlah_sebutan|
+----------+--------------+
|   Mandiri|             3|
|       BCA|             0|
|       BRI|             0|
|    Telkom|             0|
|     Astra|             0|
+----------+--------------+


📊 Distribusi sentimen keseluruhan berita:


[Stage 59:===================>                                      (1 + 2) / 3]

+--------+-----+
|sentimen|count|
+--------+-----+
|  netral|  379|
| negatif|   12|
| positif|    9|
+--------+-----+

✅ Analisis 3 selesai


In [15]:
print("💾 Menyimpan hasil analisis ke dashboard...")

output_payload = {
    "metadata": {
        "project"       : "SahamMeter",
        "kelompok"      : 3,
        "generated_at"  : datetime.now().isoformat(),
        "spark_version" : spark.version,
        "saham_dari_hdfs" : SAHAM_DARI_HDFS,
        "berita_dari_hdfs": BERITA_DARI_HDFS,
        "hdfs_base"     : HDFS_BASE,
    },
    "analisis_1_return": result_return,
    "analisis_2_volatilitas": result_volatilitas,
    "analisis_3_frekuensi_berita": result_frekuensi,
}

# Simpan ke file lokal
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2, default=str)

size_kb = OUTPUT_JSON.stat().st_size / 1024
print(f"   ✅ Tersimpan ke : {OUTPUT_JSON.resolve()}")
print(f"   📦 Ukuran file  : {size_kb:.2f} KB")
print(f"   🕐 Timestamp    : {output_payload['metadata']['generated_at']}")


💾 Menyimpan hasil analisis ke dashboard...
   ✅ Tersimpan ke : /Users/gilang/Documents/Kuliah/Semester 4/Big Data/kelompok-3-ets-bigdata/dashboard/data/spark_results.json
   📦 Ukuran file  : 6.04 KB
   🕐 Timestamp    : 2026-05-03T14:53:06.366199


In [16]:
print("☁️  Menyimpan hasil ke HDFS...")

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

def simpan_ke_hdfs(df, hdfs_path: str, label: str):
    """Simpan DataFrame ke HDFS sebagai JSON. Return True jika sukses."""
    try:
        (
            df.coalesce(1)            # 1 file output
            .write
            .mode("overwrite")
            .option("encoding", "UTF-8")
            .json(hdfs_path)
        )
        print(f"   ✅ [{label}] tersimpan ke {hdfs_path}")
        return True
    except Exception as e:
        print(f"   ❌ [{label}] gagal simpan ke HDFS: {e}")
        return False


# Definisi output HDFS per analisis
hdfs_outputs = [
    (df_return,      f"{HDFS_HASIL_DIR}return_{timestamp_str}",      "Return Saham"),
    (df_volatilitas, f"{HDFS_HASIL_DIR}volatilitas_{timestamp_str}", "Volatilitas"),
    (df_frekuensi,   f"{HDFS_HASIL_DIR}frekuensi_{timestamp_str}",   "Frekuensi Berita"),
]

hdfs_sukses = 0
for df, path, label in hdfs_outputs:
    ok = simpan_ke_hdfs(df, path, label)
    if ok:
        hdfs_sukses += 1

print(f"\n   Total tersimpan ke HDFS: {hdfs_sukses}/{len(hdfs_outputs)}")

if hdfs_sukses == 0:
    print("   ⚠️  Semua gagal ke HDFS — hasil tetap tersimpan di dashboard lokal")

☁️  Menyimpan hasil ke HDFS...


   ✅ [Return Saham] tersimpan ke hdfs://100.74.49.87:8020/data/saham/hasil/return_20260503_145328


   ✅ [Volatilitas] tersimpan ke hdfs://100.74.49.87:8020/data/saham/hasil/volatilitas_20260503_145328


   ✅ [Frekuensi Berita] tersimpan ke hdfs://100.74.49.87:8020/data/saham/hasil/frekuensi_20260503_145328

   Total tersimpan ke HDFS: 3/3


In [18]:
print("═" * 60)
print("  RINGKASAN HASIL — SahamMeter ETS Big Data Kelompok 3")
print("═" * 60)

print(f"\n📌 Sumber Data:")
print(f"   Saham  : {'HDFS ✅' if SAHAM_DARI_HDFS else 'DUMMY ⚠️'}")
print(f"   Berita : {'HDFS ✅' if BERITA_DARI_HDFS else 'DUMMY ⚠️'}")

print(f"\n📊 Analisis 1 — Return per Saham:")
for r in result_return:
    arrow = "▲" if r["return_pct"] > 0 else ("▼" if r["return_pct"] < 0 else "─")
    print(f"   {r['ticker']:6s} {arrow} {r['return_pct']:+.2f}%  "
          f"(Rp{r['harga_awal']:,.0f} → Rp{r['harga_terkini']:,.0f})")

print(f"\n📊 Analisis 2 — Volatilitas Intraday:")
for r in result_volatilitas:
    print(f"   {r['ticker']:6s}  stddev={r['stddev_harga']:.2f}  "
          f"range={r['range_intraday']:.0f}  [{r['level_volatilitas']}]")

print(f"\n📊 Analisis 3 — Frekuensi Sebutan Berita:")
for r in result_frekuensi:
    bar = "█" * r["jumlah_sebutan"]
    bar = bar if bar else "─"
    print(f"   {r['perusahaan']:8s} {bar} {r['jumlah_sebutan']}×")

print(f"\n💾 Output:")
print(f"   Lokal  : {OUTPUT_JSON.resolve()}")
print(f"   HDFS   : {HDFS_HASIL_DIR}")
print(f"\n✅ Semua analisis selesai — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("═" * 60)

════════════════════════════════════════════════════════════
  RINGKASAN HASIL — SahamMeter ETS Big Data Kelompok 3
════════════════════════════════════════════════════════════

📌 Sumber Data:
   Saham  : HDFS ✅
   Berita : HDFS ✅

📊 Analisis 1 — Return per Saham:
   INDF   ▲ +2.43%  (Rp7,131 → Rp7,304)
   ASII   ▲ +2.00%  (Rp5,148 → Rp5,251)
   ICBP   ▲ +0.95%  (Rp11,546 → Rp11,655)
   TLKM   ▲ +0.55%  (Rp3,874 → Rp3,895)
   GOTO   ▲ +0.27%  (Rp88 → Rp88)
   BMRI   ▼ -0.17%  (Rp6,144 → Rp6,133)
   UNVR   ▼ -0.42%  (Rp2,105 → Rp2,096)
   PGAS   ▼ -0.90%  (Rp1,461 → Rp1,448)
   BBRI   ▼ -1.05%  (Rp4,809 → Rp4,759)
   BBCA   ▼ -1.80%  (Rp9,588 → Rp9,416)

📊 Analisis 2 — Volatilitas Intraday:
   PGAS    stddev=27.05  range=70  [SEDANG]
   ICBP    stddev=165.59  range=526  [SEDANG]
   INDF    stddev=94.36  range=276  [SEDANG]
   UNVR    stddev=26.34  range=98  [SEDANG]
   TLKM    stddev=45.09  range=150  [SEDANG]
   BBCA    stddev=99.91  range=352  [SEDANG]
   BBRI    stddev=25.62  range=1